# B1 — The folding predictor, tested on SH3 families it never saw

**Question.** Can ΔΔG of folding be predicted for an SH3 domain from sequence alone, and does a
supervised model beat the zero-shot ESM-2 score it is built on?

**What it does.** A 1292 → 512 → 128 → 1 network on frozen ESM-2 features, trained on the
~361,000 **non-SH3** MegaScale variants (validation = whole held-out domains), five seeds. The
entire SH3 family — 15 genes, 27,590 variants — is held out, and each domain is scored
separately with a 2,000-resample bootstrap interval. Splits are grouped on UniProt accession +
domain index so no domain sits on both sides.

**Inputs:** A2 features and embeddings; A1 ground truth (for the cross-assay check).
**Outputs:** `models.pt`, `folding_per_domain_ci.csv`, `heldout_predictions.parquet`,
`atlas_fold_predictions.parquet`, `ablation.csv`, `RESULTS_B1.md`.
It also runs four robustness checks (feature ablation, ridge regression, error breakdown, FoldX/EVE),
which do not change the model.
**Runtime:** about 25 min on a **T4 GPU** (10 of them the robustness checks). Explanation: `docs/B1_folding_model.md`.

In [ ]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'B1', 'Folding predictor', resume=RESUME)
if C.DEV == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
print(C.env_info())

In [ ]:
import torch
MG = pd.read_parquet(R.res('A2', 'features_megascale.parquet'))
DD = pd.read_parquet(R.res('A2', 'features_ddpca.parquet'))
AT = pd.read_parquet(R.res('A2', 'features_atlas.parquet'))
EMB = C.load_embeddings(R.res('A2', 'esm_domain_embeddings.npz'))
print(f'megascale {len(MG):,} | ddpca {len(DD):,} | atlas {len(AT):,} | embeddings {len(EMB)}')

## 1. Train on non-SH3 domains (or reload this folder's models)

In [ ]:
Xall, M = C.build_X(MG, EMB)
M['group'] = np.where(M.uniprot.notna() & M.domain_idx.notna(),
                      M.uniprot.astype(str) + '_' + M.domain_idx.astype(str), M.domain_key.astype(str))
tr = np.where(~M.is_sh3.eq(True).values)[0]
g = M.group.values[tr]; uq = np.unique(g); np.random.default_rng(0).shuffle(uq)
vm = np.isin(g, uq[:max(1, len(uq) // 10)]); fit, val = tr[~vm], tr[vm]
SPLIT = dict(train_variants=len(fit), train_groups=len(np.unique(M.group.values[fit])),
             val_variants=len(val), val_groups=len(np.unique(M.group.values[val])),
             heldout_sh3_variants=int(M.is_sh3.eq(True).sum()))
assert not set(M.group.values[fit]) & set(M.group.values[M.is_sh3.eq(True).values]), 'SH3 group leaked into training'
print(SPLIT)
if R.done('models.pt'):
    FOLD, SC_M, SC_SD = C.load_fold_models(R.out('models.pt')); print('folding models loaded from this folder')
else:
    Xf = Xall[fit].astype(np.float32); SC_M = Xf.mean(0); SC_SD = Xf.std(0) + 1e-6
    Xf = (Xf - SC_M) / SC_SD; Xv = (Xall[val].astype(np.float32) - SC_M) / SC_SD
    y = M.ddG_fold.values.astype(np.float32); FOLD = []
    for s in range(5):
        t0 = time.time(); FOLD.append(C.train(C.FoldHead, Xf, y[fit], Xv, y[val], epochs=30, bs=1024, wd=1e-4, seed=s))
        print(f'  seed {s}  {(time.time() - t0) / 60:.1f} min')
    del Xf, Xv
    torch.save({'fold': [m.state_dict() for m in FOLD], 'fold_scaler': {'m': SC_M, 'sd': SC_SD}}, R.out('models.pt'))
del Xall

## 2. Score every held-out SH3 domain separately
Per-domain ρ with a bootstrap interval, the zero-shot ESM-2 score on the same variants, and a sign test across domains. Predictions for the calibration step (B3) are saved alongside.

In [ ]:
X32, T = C.build_X(MG[MG.is_sh3.eq(True)], EMB, half=False)
T['group'] = T.uniprot.astype(str) + '_' + T.domain_idx.astype(str)
T['pred'] = C.fold_scores(FOLD, X32, SC_M, SC_SD, clip=20).mean(0)            # Stage 4 evaluation setting
P10 = C.fold_scores(FOLD, X32.astype(np.float16), SC_M, SC_SD, clip=10)        # Stage 3b calibration setting
T['pred_c10'], T['sd_c10'] = P10.mean(0), P10.std(0)
rows = []
for g_, sub in T.groupby('group'):
    if len(sub) < 30: continue
    y, p = sub.ddG_fold.values, sub.pred.values; lo, hi = C.boot_ci(y, p, 2000)
    rows.append(dict(group=g_, gene=sub.gene.iloc[0], n=len(sub), rho=C.spearman(y, p), lo=lo, hi=hi,
                     esm=C.spearman(y, sub.esm_score.values)))
PD = pd.DataFrame(rows).sort_values('rho'); PD.to_csv(R.out('folding_per_domain_ci.csv'), index=False)
WINS = int((PD.rho > PD.esm.abs()).sum())
T[['domain_key', 'group', 'gene', 'uniprot', 'domain_idx', 'pos', 'wt_aa', 'mut_aa', 'ddG_fold', 'esm_score',
   'pred', 'pred_c10', 'sd_c10']].to_parquet(R.out('heldout_predictions.parquet'), index=False)
print(PD.round(3).to_string(index=False))
print(f'\nper-domain median rho {PD.rho.median():.3f} | ESM {PD.esm.median():.3f} | head wins {WINS}/{len(PD)} '
      f'| sign test p = {C.sign_test_two_sided(WINS, len(PD)):.1e}')

## 3. Apply to the atlas, and a cross-assay check on ddPCA folding energies

In [ ]:
Xa, A = C.build_X(AT, EMB)
Pa = C.fold_scores(FOLD, Xa, SC_M, SC_SD, clip=10)
A['pred_ddG_fold'], A['sd_fold'] = Pa.mean(0), Pa.std(0)
A.to_parquet(R.out('atlas_fold_predictions.parquet'), index=False)
rel = pd.read_parquet(R.res('A1', 'ddpca_ground_truth.parquet'))[['domain', 'pos', 'wt_aa', 'mut_aa', 'folding_reliable']]
Xd, D_ = C.build_X(DD, EMB)
D_['pred'] = C.fold_scores(FOLD, Xd, SC_M, SC_SD, clip=20).mean(0)
D_ = D_.merge(rel.rename(columns={'domain': 'domain_key'}), on=['domain_key', 'pos', 'wt_aa', 'mut_aa'], how='left')
XA = pd.DataFrame([dict(domain=k, n=len(g_), rho_model=C.spearman(g_.ddG_fold, g_.pred), rho_esm=C.spearman(g_.ddG_fold, g_.esm_score))
                   for k, g_ in D_[D_.folding_reliable.eq(True) & D_.ddG_fold.notna()].groupby('domain_key')])
print(XA.round(3).to_string(index=False))

## 4. Robustness checks
- **Feature ablation (roadmap K1):** what does each feature group contribute? One seed each, same split.
- **Simpler models (K2):** ridge regression on the same 1,292 features.
- **Error breakdown (J4):** where the head is wrong — by substitution type and by effect size.
- **Conventional predictors (J6):** FoldX and EVE (shipped with the ddPCA archive) against the head on the ddPCA domains.
None of these changes the model or the headline numbers above.

In [ ]:
from sklearn.linear_model import Ridge
pdmed = lambda p: float(pd.Series([C.spearman(g_.ddG_fold, p[g_.index]) for _, g_ in T.groupby('group') if len(g_) >= 30]).median())
if R.done('ablation.csv'):
    AB = pd.read_csv(R.out('ablation.csv')); print('ablation loaded from this folder')
else:
    Xall, _ = C.build_X(MG, EMB); y = M.ddG_fold.values.astype(np.float32); rows = []
    for name, cols in [('ESM embedding only', list(range(1280))), ('embedding + 5 ESM scalars', list(range(1285))),
                       ('12 scalars only (no embedding)', list(range(1280, 1292)))]:
        Xf = Xall[fit][:, cols].astype(np.float32); m_ = Xf.mean(0); s_ = Xf.std(0) + 1e-6
        mdl = C.train(C.FoldHead, (Xf - m_) / s_, y[fit], (Xall[val][:, cols].astype(np.float32) - m_) / s_, y[val],
                      epochs=30, bs=1024, wd=1e-4, seed=0)
        p = C.predict(mdl, np.clip((X32[:, cols] - m_) / s_, -20, 20)); del Xf
        rows.append(dict(model=f'MLP, {name}', features=len(cols), per_domain_median_rho=pdmed(p), pooled_rho=C.spearman(T.ddG_fold, p)))
        print(rows[-1])
    p = C.predict(FOLD[0], np.clip((X32 - SC_M) / SC_SD, -20, 20))
    rows.append(dict(model='MLP, all 1,292 features (seed 0)', features=1292, per_domain_median_rho=pdmed(p), pooled_rho=C.spearman(T.ddG_fold, p)))
    rows.append(dict(model='MLP, all features, 5-seed ensemble (the model)', features=1292, per_domain_median_rho=pdmed(T.pred.values),
                     pooled_rho=C.spearman(T.ddG_fold, T.pred)))
    Xs = ((Xall[fit].astype(np.float32) - SC_M) / SC_SD); Xv = ((Xall[val].astype(np.float32) - SC_M) / SC_SD)
    best = max([1, 10, 100, 1000], key=lambda a: C.spearman(y[val], Ridge(alpha=a).fit(Xs, y[fit]).predict(Xv)))
    rg = Ridge(alpha=best).fit(Xs, y[fit]); p = rg.predict(np.clip((X32 - SC_M) / SC_SD, -20, 20)); del Xs, Xv, Xall
    rows.append(dict(model=f'ridge regression, all features (alpha {best})', features=1292, per_domain_median_rho=pdmed(p),
                     pooled_rho=C.spearman(T.ddG_fold, p)))
    rows.append(dict(model='zero-shot ESM-2 score (no training)', features=0, per_domain_median_rho=pdmed(-T.esm_score.values),
                     pooled_rho=C.spearman(T.ddG_fold, -T.esm_score)))
    AB = pd.DataFrame(rows); AB.to_csv(R.out('ablation.csv'), index=False)
print(AB.round(3).to_string(index=False))

In [ ]:
KDm = T.mut_aa.map(C.KD); KDw = T.wt_aa.map(C.KD)
T['substitution'] = np.select([T.mut_aa == 'P', T.wt_aa == 'G', T.mut_aa == 'G', (KDw > 1.5) & (KDm < -1.5), (KDw < -1.5) & (KDm > 1.5)],
                              ['to proline', 'from glycine', 'to glycine', 'hydrophobic to polar', 'polar to hydrophobic'], 'other')
T['effect'] = pd.cut(T.ddG_fold, [-np.inf, 0, 1, 2, 3, np.inf], labels=['stabilising (<0)', '0-1', '1-2', '2-3', '>3 kcal/mol'])
def err_table(col):
    return T.groupby(col, observed=True).apply(lambda g_: pd.Series(dict(n=len(g_), rho=C.spearman(g_.ddG_fold, g_.pred) if len(g_) > 20 else np.nan,
                                                            mae=float(np.mean(np.abs(g_.ddG_fold - g_.pred))),
                                                            bias=float(np.mean(g_.pred - g_.ddG_fold))))).reset_index()
ERR_SUB, ERR_EFF = err_table('substitution'), err_table('effect')
print(ERR_SUB.round(3).to_string(index=False)); print(ERR_EFF.round(3).to_string(index=False))

import re
fx = pd.read_csv(R.res('A1', 'baseline_foldx.csv')); ev = pd.read_csv(R.res('A1', 'baseline_eve.csv'))
D2 = D_[D_.folding_reliable.eq(True) & D_.ddG_fold.notna()].copy(); D2['id'] = D2.wt_aa + D2.pos.astype(int).astype(str) + D2.mut_aa
fx = fx.join(fx.id.astype(str).str.extract(r'^([A-Z])(\d+)([A-Z])$').rename(columns={0: 'f_wt', 1: 'f_pos', 2: 'f_mut'})).dropna(subset=['f_pos'])
fx['f_pos'] = fx.f_pos.astype(int); fparts = []
for dom, g_ in fx.groupby('domain'):         # FoldX numbers residues as in the PDB file (GB1 +1, PDZ3 +310): derive the offset
    wt_at = dict(zip(D2[D2.domain_key == dom].pos.astype(int), D2[D2.domain_key == dom].wt_aa))
    off = max(range(-400, 401), key=lambda o: sum(wt_at.get(p - o) == w for p, w in zip(g_.f_pos, g_.f_wt)))
    fparts.append(g_.assign(domain_key=dom, pos=g_.f_pos - off, mut_aa=g_.f_mut, foldx_offset=off))
fx = pd.concat(fparts)[['domain_key', 'pos', 'mut_aa', 'foldx_ddg', 'foldx_offset']]
D2 = D2.merge(fx, on=['domain_key', 'pos', 'mut_aa'], how='left')
_f = D2.dropna(subset=['foldx_ddg'])
FOLDX_R = dict(foldx_overlap_aligned=len(_f), foldx_vs_mochi_r_aligned=float(_f.ddG_fold.corr(_f.foldx_ddg)),
               **{f'foldx_vs_mochi_r_aligned_{d}': float(g.ddG_fold.corr(g.foldx_ddg)) for d, g in _f.groupby('domain_key')})
print(FOLDX_R)
ev = ev.join(ev.id.astype(str).str.extract(r'^([A-Z])(\d+)([A-Z])$').rename(columns={0: 'e_wt', 1: 'e_pos', 2: 'e_mut'})).dropna(subset=['e_pos'])
ev['e_pos'] = ev.e_pos.astype(int); parts = []
for dom, g_ in ev.groupby('domain'):         # EVE numbers residues on the full protein: derive the offset
    wt_at = dict(zip(D2[D2.domain_key == dom].pos.astype(int), D2[D2.domain_key == dom].wt_aa))
    off = max(range(-400, 401), key=lambda o: sum(wt_at.get(p - o) == w for p, w in zip(g_.e_pos, g_.e_wt)))
    parts.append(g_.assign(domain_key=dom, pos=g_.e_pos - off, mut_aa=g_.e_mut, eve_offset=off))
ev = pd.concat(parts)[['domain_key', 'pos', 'mut_aa', 'prediction_EVE_evol_indices', 'eve_offset']]
D2 = D2.merge(ev, on=['domain_key', 'pos', 'mut_aa'], how='left')
BASE = []
for dom, g_ in D2.groupby('domain_key'):
    c = g_.dropna(subset=['foldx_ddg', 'prediction_EVE_evol_indices'])
    BASE.append(dict(domain=dom, common_variants=len(c), head=C.spearman(c.ddG_fold, c.pred), esm_zero_shot=C.spearman(c.ddG_fold, -c.esm_score),
                     foldx=C.spearman(c.ddG_fold, c.foldx_ddg), eve=C.spearman(c.ddG_fold, c.prediction_EVE_evol_indices),
                     foldx_offset=int(g_.foldx_offset.dropna().iloc[0]) if g_.foldx_offset.notna().any() else None,
                     eve_offset=int(g_.eve_offset.dropna().iloc[0]) if g_.eve_offset.notna().any() else None))
BASE = pd.DataFrame(BASE); print(BASE.round(3).to_string(index=False))

## 5. Results

In [ ]:
from scipy.stats import pearsonr
q1, q3 = np.percentile(PD.rho, [25, 75])
for k, v in [('domains', len(PD)), ('per_domain_median_rho', PD.rho.median()), ('per_domain_min_rho', PD.rho.min()),
             ('per_domain_max_rho', PD.rho.max()), ('per_domain_iqr_low', q1), ('per_domain_iqr_high', q3),
             ('all_ci_exclude_zero', bool((PD.lo > 0).all())), ('esm_per_domain_median_rho', PD.esm.median()),
             ('domains_won', WINS), ('sign_test_p', C.sign_test_two_sided(WINS, len(PD))),
             ('heldout_variants', len(T)), ('pooled_rho', C.spearman(T.ddG_fold, T.pred)),
             ('pooled_pearson_r', float(pearsonr(T.ddG_fold, T.pred)[0])),
             ('pooled_rmse', float(np.sqrt(np.mean((T.ddG_fold - T.pred) ** 2)))),
             *[(k, v) for k, v in SPLIT.items()]]:
    R.metric(k, v)
R.section('Per held-out SH3 domain', 'Spearman ρ against measured ΔΔG_fold, 95% bootstrap CI (2,000 resamples). '
          'ESM is the zero-shot score on the same variants (negative sign is correct).', PD, '{:.3f}')
R.section('Cross-assay check on ddPCA folding energies (reliable fits)', 'A different assay (AbundancePCA) from the '
          'training data (proteolysis); GRB2-SH3 is the only SH3.', XA, '{:.3f}')
for _, r in AB.iterrows(): R.metric('ablation: ' + r.model, r.per_domain_median_rho)
for _, r in BASE.iterrows():
    for c in ('head', 'esm_zero_shot', 'foldx', 'eve'): R.metric(f'baseline_{r.domain}_{c}', r[c])
for k, v in FOLDX_R.items(): R.metric(k, v)
R.section('Feature ablation and simpler models (K1, K2)', 'Per-domain median ρ on the 15 held-out SH3 domains. Single-seed rows '
          'should be compared with the single-seed full model, not the ensemble.', AB, '{:.3f}')
R.section('Error by substitution type (J4)', 'bias = mean(predicted − measured); negative means under-predicted damage.', ERR_SUB, '{:.3f}')
R.section('Error by measured effect size (J4)', '', ERR_EFF, '{:.3f}')
R.section('Against conventional predictors on the ddPCA domains (J6)', 'Spearman ρ with measured ΔΔG_fold on variants all four '
          'methods score. FoldX needs a structure; EVE needs a deep alignment; the head needs neither. FoldX (PDB numbering) and EVE '
          'residue numbers are converted to construct numbering by offsets derived from the wild-type residues.', BASE, '{:.3f}')
R.finish([f"Per-domain median ρ {PD.rho.median():.3f} (range {PD.rho.min():.3f}–{PD.rho.max():.3f}, IQR {q1:.3f}–{q3:.3f}) "
          f"over {len(PD)} held-out SH3 domains; every 95% CI excludes zero: {bool((PD.lo > 0).all())}.",
          f"Beats zero-shot ESM-2 (median |ρ| {PD.esm.abs().median():.3f}) on {WINS}/{len(PD)} domains; "
          f"two-sided sign test p = {C.sign_test_two_sided(WINS, len(PD)):.1e}.",
          f"Pooled over {len(T):,} variants: ρ {R.metrics['pooled_rho']:.3f}, RMSE {R.metrics['pooled_rmse']:.3f} kcal/mol "
          f"(quote the per-domain figure: SPTAN1 alone contributes {int((T.gene == 'SPTAN1').sum()):,} variants).",
          'The model uses no structural feature; structure is kept for the independent check in B2.'])